# 🚀 ViT AI Image Detector 训练

**运行前确保：** Runtime → Change runtime type → **A100 GPU**

预计训练时间: **30-45 分钟** (50K图片, 5 epochs)

## 1️⃣ 检查 GPU

In [6]:
!nvidia-smi

Sat Jan 24 19:16:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2️⃣ 安装依赖

In [2]:
!pip install -q torch torchvision transformers scikit-learn tqdm pillow kagglehub mlflow huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.3/788.3 kB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 21.1 MB/s eta 0:00:00


## 3️⃣ 克隆仓库

In [3]:
# 克隆你的仓库
!git clone https://github.com/wanikua/ItsNotAI-Model-Training.git
%cd ItsNotAI-Model-Training

print('✅ 仓库克隆成功!')

Cloning into 'ItsNotAI-Model-Training'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 98 (delta 27), reused 98 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 6.35 MiB | 29.81 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/ItsNotAI-Model-Training
✅ 仓库克隆成功!


In [4]:
# 添加项目到 Python 路径
import sys
sys.path.insert(0, '/content/ItsNotAI-Model-Training')

# 验证导入
from src.models.vit_detector import ViTDetector
print('✅ 项目导入成功!')

✅ 项目导入成功!


## 4️⃣ 下载数据集

In [7]:
from src.dataset.download_datasets import prepare_combined_dataset, verify_dataset

DATA_ROOT = '/content/data'
prepare_combined_dataset(DATA_ROOT)
verify_dataset(DATA_ROOT)

AttributeError: 'str' object has no attribute 'mkdir'

## 5️⃣ 开始训练! 🎯

In [ ]:
from src.training.config import TrainingConfig
from src.training.train_vit import Trainer
from pathlib import Path

# 多分类配置 - 识别具体生成器
config = TrainingConfig.for_colab_multiclass()
config.data_root = Path('/content/data')
config.output_dir = Path('/content/outputs')
config.num_epochs = 10
config.batch_size = 64  # T4 用 64, A100 可以用 128
config.learning_rate = 5e-6
config.use_mlflow = False  # Colab 里禁用 MLflow

print('配置:')
print(f'  模式: {"多分类 (识别生成器)" if config.multiclass else "二分类 (Real/Fake)"}')
print(f'  Batch size: {config.batch_size}')
print(f'  Epochs: {config.num_epochs}')
print(f'  Learning rate: {config.learning_rate}')

In [ ]:
# 开始训练!
trainer = Trainer(config)
test_metrics = trainer.train()

print('\n🎉 训练完成!')
print(f'测试集指标: {test_metrics}')

## 6️⃣ 保存模型到 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 保存模型
!mkdir -p "/content/drive/MyDrive/AI-Models"
!cp -r /content/outputs/best_model "/content/drive/MyDrive/AI-Models/vit-ai-detector"

print('✅ 模型已保存到 Google Drive: AI-Models/vit-ai-detector')

## 7️⃣ 测试模型

In [ ]:
from src.models.vit_detector import ViTDetector
from PIL import Image
import requests
from io import BytesIO

# 加载训练好的模型
model = ViTDetector.load('/content/outputs/best_model')

# 测试一张图片
def test_url(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert('RGB')
    
    # 多分类预测
    result = model.predict_multiclass(img, top_k=5)
    
    print(f'预测来源: {result.predicted_source}')
    print(f'是否真实: {"✅ 真实图片" if result.is_real else "❌ AI生成"}')
    print(f'置信度: {result.confidence:.2%}')
    print(f'\nTop 5 可能的来源:')
    for source, prob in result.top_k:
        marker = "📷" if model.source_is_real.get(source, False) else "🤖"
        print(f'  {marker} {source}: {prob:.2%}')
    
    # 综合判断
    real_prob, fake_prob = model.get_real_vs_fake_prob(img)
    print(f'\n综合判断: Real={real_prob:.2%}, Fake={fake_prob:.2%}')
    
    return result

# 替换为你想测试的图片 URL
# test_url('https://example.com/test_image.jpg')

In [ ]:
# 上传本地图片测试
from google.colab import files

print('上传一张图片测试:')
uploaded = files.upload()

for filename in uploaded.keys():
    img = Image.open(filename).convert('RGB')
    result = model.predict_multiclass(img, top_k=5)
    
    print(f'\n{"="*50}')
    print(f'文件: {filename}')
    print(f'{"="*50}')
    print(f'预测来源: {result.predicted_source}')
    print(f'是否真实: {"✅ 真实图片" if result.is_real else "❌ AI生成"}')
    print(f'置信度: {result.confidence:.2%}')
    print(f'\nTop 5 可能的来源:')
    for source, prob in result.top_k:
        marker = "📷" if model.source_is_real.get(source, False) else "🤖"
        print(f'  {marker} {source}: {prob:.2%}')
    
    real_prob, fake_prob = model.get_real_vs_fake_prob(img)
    print(f'\n综合判断: Real={real_prob:.2%}, Fake={fake_prob:.2%}')